# Evershed (1909) — Radial Movement in Sun-Spots: Implementation
# 흑점에서의 방사상 운동: 구현

**Paper / 논문**: J. Evershed, "Radial Movement in Sun-Spots," *MNRAS*, 69, 454–457, 1909.

이 노트북은 Evershed 효과의 핵심 물리학을 구현합니다:

1. **Part 1**: Doppler 이동 기하학 — 방사 vs. 원형 운동 시뮬레이션 / Doppler shift geometry
2. **Part 2**: Line-of-Nodes 분석 — Evershed의 Table 재현 / Line-of-nodes analysis
3. **Part 3**: 합성 흑점 스펙트럼 — Evershed 흐름에 의한 선 기울기 / Synthetic sunspot spectrum
4. **Part 4**: Evershed 흐름 속도장 시각화 / Evershed flow velocity field
5. **Part 5**: Hale의 자기장 + Evershed 흐름 = MHD 구조 / Combined magnetic + flow structure

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, FancyArrowPatch
from matplotlib.gridspec import GridSpec

c_km = 2.998e5  # Speed of light in km/s

plt.rcParams.update({
    'figure.figsize': (12, 6),
    'font.size': 12,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

## Part 1: Doppler Shift Geometry — Radial vs. Circular Motion
## 파트 1: Doppler 이동 기하학 — 방사 vs. 원형 운동

흑점이 태양 원반 위 특정 위치에 있을 때, 방사 운동과 원형(소용돌이) 운동이 만드는
Doppler 이동 패턴의 차이를 시각화합니다. 이것이 Evershed가 두 가설을 구별한 핵심입니다.

Visualizing the difference in Doppler shift patterns between radial and circular (vortex) motion
for a sunspot at a given disk position. This is how Evershed distinguished the two hypotheses.

In [ ]:
def los_doppler_map(motion_type: str, spot_angle_deg: float,
                    v_surface: float = 1.9, n: int = 200) -> tuple:
    """Compute LOS Doppler velocity map for radial or circular motion in a sunspot.

    Args:
        motion_type: 'radial' or 'circular'.
        spot_angle_deg: Angular distance of spot from disk center (degrees).
        v_surface: Surface flow speed (km/s).
        n: Grid resolution.

    Returns:
        Tuple of (X, Y, V_los) where X, Y are in spot-radius units
        and V_los is line-of-sight velocity in km/s.
    """
    theta = np.radians(spot_angle_deg)
    r_max = 1.0  # Penumbral outer edge

    x = np.linspace(-1.2, 1.2, n)
    y = np.linspace(-1.2, 1.2, n)
    X, Y = np.meshgrid(x, y)
    R = np.sqrt(X**2 + Y**2)

    # Flow speed profile: peaks at penumbral edge, zero at center
    v_mag = v_surface * np.clip(R / r_max, 0, 1)
    v_mag[R > r_max] = 0  # Abrupt stop at penumbral boundary
    v_mag[R < 0.3] = 0    # No flow in umbra

    if motion_type == 'radial':
        # Radial outflow: velocity points away from center
        vx = v_mag * X / np.where(R > 0, R, 1)
        vy = v_mag * Y / np.where(R > 0, R, 1)
    elif motion_type == 'circular':
        # Counter-clockwise vortex: velocity tangent to circles
        vx = -v_mag * Y / np.where(R > 0, R, 1)
        vy =  v_mag * X / np.where(R > 0, R, 1)
    else:
        raise ValueError("motion_type must be 'radial' or 'circular'")

    # LOS projection: spot to the RIGHT (positive x = direction to disk center)
    # LOS component of horizontal velocity: vx * sin(theta)
    V_los = vx * np.sin(theta)

    # Mask outside penumbra
    V_los[R > r_max] = np.nan
    V_los[R < 0.3] = np.nan

    return X, Y, V_los


# --- Compare radial vs. circular for a spot 40° from center ---
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

spot_angle = 40  # degrees from disk center

for ax, mtype, title in zip(axes[:2],
    ['radial', 'circular'],
    ['Radial Outflow (Evershed)\n방사 유출 (Evershed)',
     'Circular Vortex (Hale hypothesis)\n원형 소용돌이 (Hale 가설)']):

    X, Y, V = los_doppler_map(mtype, spot_angle)
    im = ax.pcolormesh(X, Y, V, cmap='RdBu_r', vmin=-2, vmax=2, shading='auto')
    plt.colorbar(im, ax=ax, label='V_LOS (km/s)', shrink=0.8)

    # Spot structure
    ax.add_patch(Circle((0, 0), 1.0, fill=False, ec='black', lw=2, ls='--',
                         label='Penumbra edge'))
    ax.add_patch(Circle((0, 0), 0.3, fill=True, fc='gray', ec='black', lw=1.5,
                         alpha=0.5, label='Umbra'))

    # Direction to disk center
    ax.annotate('→ Disk center', xy=(0.6, -1.1), fontsize=10, color='black',
                fontweight='bold')

    # Slit directions
    ax.axhline(0, color='green', ls='-', lw=1.5, alpha=0.7)
    ax.axvline(0, color='orange', ls='-', lw=1.5, alpha=0.7)

    ax.set_xlim(-1.3, 1.3)
    ax.set_ylim(-1.3, 1.3)
    ax.set_aspect('equal')
    ax.set_title(title, fontsize=12)
    ax.set_xlabel('x (spot radii) — toward disk center →')
    ax.set_ylabel('y (spot radii)')

# Right panel: slit-position analysis
ax3 = axes[2]
slit_angles = np.linspace(0, 180, 100)

# For each slit angle, compute the max Doppler shift
def max_shift_vs_slit(motion_type, spot_angle, slit_angle_deg):
    """Approximate max Doppler shift for a slit at given angle."""
    theta = np.radians(spot_angle)
    phi = np.radians(slit_angle_deg)  # Slit angle from x-axis (disk center direction)

    if motion_type == 'radial':
        # Radial: max shift when slit along x (cos(phi))
        return 1.9 * np.sin(theta) * np.abs(np.cos(phi))
    else:
        # Circular: max shift when slit along y (sin(phi))
        return 1.9 * np.sin(theta) * np.abs(np.sin(phi))

shift_radial = [max_shift_vs_slit('radial', 40, a) for a in slit_angles]
shift_circular = [max_shift_vs_slit('circular', 40, a) for a in slit_angles]

ax3.plot(slit_angles, shift_radial, 'b-', lw=2.5, label='Radial (Evershed)')
ax3.plot(slit_angles, shift_circular, 'r--', lw=2.5, label='Circular (Hale)')
ax3.axvline(0, color='green', ls=':', alpha=0.5, label='Slit ∥ disk center dir.')
ax3.axvline(90, color='orange', ls=':', alpha=0.5, label='Slit ⊥ disk center dir.')

ax3.set_xlabel('Slit angle from spot→center direction (°)', fontsize=11)
ax3.set_ylabel('Max |Doppler shift| (km/s)', fontsize=11)
ax3.set_title('Shift vs. Slit Orientation\n선 이동 vs. Slit 방향', fontsize=12)
ax3.legend(fontsize=9)
ax3.set_xlim(0, 180)

# Annotations
ax3.annotate('Radial: max at 0°\n(slit ∥ center)',
             xy=(5, 1.15), fontsize=9, color='blue')
ax3.annotate('Circular: max at 90°\n(slit ⊥ center)',
             xy=(65, 1.15), fontsize=9, color='red')

plt.tight_layout()
plt.show()

print("Key distinction: Radial motion → max shift when slit points to disk center (0°)")
print("                 Circular motion → max shift when slit perpendicular to center (90°)")
print("                 Evershed observed max at 0° → RADIAL motion confirmed")

## Part 2: Line-of-Nodes Analysis — Reproducing Evershed's Table
## 파트 2: Line-of-Nodes 분석 — Evershed 테이블 재현

Evershed의 핵심 증거: line of nodes와 흑점–태양 중심 방향 사이의 각도가 평균 **90°**.
방사 운동이면 90°, 원형 운동이면 0°를 예측합니다.

Evershed's key evidence: the angle between line of nodes and spot–disk-center direction averages **90°**.
Radial motion predicts 90°, circular motion predicts 0°.

In [ ]:
# Evershed's Table (p. 456)
evershed_table = {
    'date':     ['Jan 18', 'Jan 18', 'Jan 20', 'Jan 25', 'Jan 25', 'Jan 26', 'Jan 27'],
    'spot_no':  [1588, 1591, 1591, 1594, 1591, 1591, 1595],
    'pa_spot':  [82, 74, 60, 87, -84, -88, 60],   # E positive, W negative
    'pa_node':  [-20, -2, -23, -8, 0, 9, -31],     # W negative, E positive
    'angle':    [102, 76, 83, 95, 84, 97, 91],      # Angle between node and center
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Left: Bar chart of angles
colors = ['steelblue', 'coral', 'coral', 'seagreen', 'coral', 'coral', 'gold']
bars = ax1.bar(range(7), evershed_table['angle'], color=colors, edgecolor='black', alpha=0.8)

ax1.axhline(90, color='blue', ls='--', lw=2.5, label='Radial prediction: 90°')
ax1.axhline(0, color='red', ls=':', lw=2, label='Vortex prediction: 0°')
mean_angle = np.mean(evershed_table['angle'])
ax1.axhline(mean_angle, color='darkgreen', ls='-', lw=2,
            label=f'Measured mean: {mean_angle:.0f}°')

ax1.set_xticks(range(7))
ax1.set_xticklabels([f"{d}\nSpot {s}" for d, s in
                      zip(evershed_table['date'], evershed_table['spot_no'])],
                     fontsize=9, rotation=0)
ax1.set_ylabel('Angle: Node–Center of Disc (°)', fontsize=12)
ax1.set_title("Evershed's Table: Line-of-Nodes Angles\nEvershed의 테이블: Line-of-Nodes 각도",
              fontsize=13)
ax1.legend(fontsize=10, loc='lower right')
ax1.set_ylim(0, 120)

# Add value labels on bars
for bar, val in zip(bars, evershed_table['angle']):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2,
             f'{val}°', ha='center', fontsize=10, fontweight='bold')

# Right: Polar plot showing spot P.A. and node P.A.
ax2 = plt.subplot(122, projection='polar')
ax2.set_theta_zero_location('N')
ax2.set_theta_direction(-1)

for i in range(7):
    pa_s = np.radians(evershed_table['pa_spot'][i])
    pa_n = np.radians(evershed_table['pa_node'][i])

    ax2.plot([pa_s, pa_s], [0, 0.8], 'o-', color=colors[i], lw=2, ms=8,
             label=f"Spot {evershed_table['spot_no'][i]}" if i < 4 else None)
    ax2.plot([pa_n, pa_n], [0, 0.5], 's-', color=colors[i], lw=1.5, ms=6,
             alpha=0.6)

ax2.set_title('Position Angles\n(● = Spot P.A., ■ = Node P.A.)', fontsize=11, pad=20)
ax2.set_ylim(0, 1)
ax2.set_yticks([])

plt.tight_layout()
plt.show()

# Statistical summary
angles = np.array(evershed_table['angle'])
print(f"=== Line-of-Nodes Analysis ===")
print(f"  Mean angle: {np.mean(angles):.1f}°")
print(f"  Std dev:    {np.std(angles):.1f}°")
print(f"  Min/Max:    {np.min(angles)}° / {np.max(angles)}°")
print(f"  Radial prediction: 90° ← MATCH")
print(f"  Vortex prediction:  0° ← REJECTED")

## Part 3: Synthetic Sunspot Spectrum — Line Inclination
## 파트 3: 합성 흑점 스펙트럼 — 선 기울기

Evershed가 관측한 것을 시뮬레이션합니다: spectrograph slit이 흑점 반암부를 지날 때,
Fraunhofer 흡수선이 Doppler shift에 의해 기울어지는 모습.

Simulating what Evershed observed: when the slit crosses the sunspot penumbra,
Fraunhofer absorption lines become tilted due to Doppler shifts.

In [ ]:
def synthetic_spectrum_2d(spot_angle_deg: float = 40.0,
                         v_max: float = 1.9,
                         lam0: float = 4850.0) -> None:
    """Generate a synthetic 2D spectrum showing Evershed line tilts.

    The vertical axis represents position along the slit (crossing the sunspot),
    and the horizontal axis represents wavelength.

    Args:
        spot_angle_deg: Angular distance of spot from disk center.
        v_max: Maximum surface flow speed (km/s).
        lam0: Central wavelength of absorption line (Å).
    """
    theta = np.radians(spot_angle_deg)
    n_pos = 200   # Spatial positions along slit
    n_lam = 300   # Wavelength points

    # Slit positions: -1.2 to +1.2 spot radii
    y_slit = np.linspace(-1.2, 1.2, n_pos)

    # Wavelength range: ±0.15 Å around line center
    lam = np.linspace(lam0 - 0.15, lam0 + 0.15, n_lam)
    LAM, Y = np.meshgrid(lam, y_slit)

    # Radial velocity profile along slit (slit aligned with disk center direction)
    R = np.abs(Y)
    v_profile = np.zeros_like(Y[:, 0])
    mask_pen = (R[:, 0] >= 0.3) & (R[:, 0] <= 1.0)
    v_profile[mask_pen] = v_max * (R[mask_pen, 0] / 1.0) * np.sign(Y[mask_pen, 0])
    v_los = v_profile * np.sin(theta)

    # Doppler-shifted line center for each position
    delta_lam = lam0 * v_los / c_km
    lam_shifted = lam0 + delta_lam

    # Build 2D spectrum
    sigma_line = 0.02  # Line width
    sigma_cont = 0.005

    spectrum = np.ones_like(LAM)  # Continuum = 1

    # Add three absorption lines
    for offset in [-0.08, 0.0, 0.08]:
        line_center = (lam0 + offset) + delta_lam[:, np.newaxis]
        depth = 0.7 if offset == 0 else 0.4
        spectrum -= depth * np.exp(-0.5 * ((LAM - line_center) / sigma_line)**2)

    # Darken umbra region
    umbra_mask = R[:, 0] < 0.3
    spectrum[umbra_mask, :] *= 0.3

    # Darken penumbra slightly
    pen_factor = np.ones(n_pos)
    pen_factor[mask_pen] = 0.7
    spectrum *= pen_factor[:, np.newaxis]

    # Plot
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 7),
                                     gridspec_kw={'width_ratios': [2, 1]})

    ax1.imshow(spectrum, extent=[lam[0], lam[-1], y_slit[-1], y_slit[0]],
               aspect='auto', cmap='gray', vmin=0, vmax=1.1)

    # Mark regions
    ax1.axhline(0.3, color='cyan', ls='--', lw=1, alpha=0.7)
    ax1.axhline(-0.3, color='cyan', ls='--', lw=1, alpha=0.7)
    ax1.axhline(1.0, color='yellow', ls='--', lw=1, alpha=0.7)
    ax1.axhline(-1.0, color='yellow', ls='--', lw=1, alpha=0.7)

    ax1.text(lam0 + 0.11, 0, 'Umbra', color='cyan', fontsize=10, va='center')
    ax1.text(lam0 + 0.11, 0.65, 'Penumbra', color='yellow', fontsize=10, va='center')
    ax1.text(lam0 + 0.11, -0.65, 'Penumbra', color='yellow', fontsize=10, va='center')

    ax1.set_xlabel('Wavelength (Å)', fontsize=12)
    ax1.set_ylabel('Position along slit (spot radii)\n← Following    Preceding →', fontsize=11)
    ax1.set_title(f'Synthetic 2D Spectrum: Evershed Line Tilts\n'
                  f'합성 2D 스펙트럼: Evershed 선 기울기\n'
                  f'(spot at {spot_angle_deg}° from center, v_max = {v_max} km/s)',
                  fontsize=12)

    # Right panel: velocity profile
    ax2.plot(v_los, y_slit, 'b-', lw=2.5)
    ax2.axvline(0, color='gray', ls='--', alpha=0.5)
    ax2.axhline(0.3, color='cyan', ls='--', lw=1, alpha=0.7)
    ax2.axhline(-0.3, color='cyan', ls='--', lw=1, alpha=0.7)
    ax2.axhline(1.0, color='yellow', ls='--', lw=1, alpha=0.7)
    ax2.axhline(-1.0, color='yellow', ls='--', lw=1, alpha=0.7)
    ax2.fill_betweenx(y_slit, 0, v_los, where=(v_los > 0), color='red', alpha=0.2,
                       label='Redshift (receding)')
    ax2.fill_betweenx(y_slit, 0, v_los, where=(v_los < 0), color='blue', alpha=0.2,
                       label='Blueshift (approaching)')
    ax2.set_xlabel('LOS Velocity (km/s)', fontsize=12)
    ax2.set_ylabel('Position along slit', fontsize=11)
    ax2.set_title('Velocity Profile\n속도 프로파일', fontsize=12)
    ax2.legend(fontsize=9)
    ax2.invert_yaxis()

    plt.tight_layout()
    plt.show()


synthetic_spectrum_2d(spot_angle_deg=40, v_max=1.9, lam0=4850)
print("The tilted absorption lines are exactly what Evershed observed —")
print("'lines crossing the spot-band were found to be slightly bent,")
print("or inclined one or two degrees' (p. 454)")

## Part 4: Evershed Flow Velocity Field
## 파트 4: Evershed 흐름 속도장 시각화

반암부 내 Evershed 흐름의 2D 속도장을 시각화합니다.
속도는 중심에서 0, 반암부 외곽에서 최대이며, 경계에서 갑자기 멈춥니다.

Visualizing the 2D velocity field of the Evershed flow in the penumbra.
Velocity is zero at center, maximum at penumbral edge, and abruptly stops at the boundary.

In [ ]:
# Evershed flow velocity field + Hale's magnetic field combined
fig, axes = plt.subplots(1, 3, figsize=(18, 5.5))

n = 200
x = np.linspace(-1.5, 1.5, n)
y = np.linspace(-1.5, 1.5, n)
X, Y = np.meshgrid(x, y)
R = np.sqrt(X**2 + Y**2)

# --- Panel 1: Evershed radial flow ---
ax = axes[0]
v_mag = 1.9 * np.clip(R, 0.3, 1.0) / 1.0
v_mag[R < 0.3] = 0
v_mag[R > 1.0] = 0

Vx = v_mag * X / np.where(R > 0, R, 1)
Vy = v_mag * Y / np.where(R > 0, R, 1)

speed = np.sqrt(Vx**2 + Vy**2)
speed[R > 1.0] = np.nan
speed[R < 0.3] = np.nan

im = ax.pcolormesh(X, Y, speed, cmap='YlOrRd', vmin=0, vmax=2.0, shading='auto')
plt.colorbar(im, ax=ax, label='|v| (km/s)', shrink=0.8)

skip = 12
mask_q = (R[::skip, ::skip] >= 0.3) & (R[::skip, ::skip] <= 1.0)
Vx_q = Vx[::skip, ::skip].copy()
Vy_q = Vy[::skip, ::skip].copy()
Vx_q[~mask_q] = np.nan
Vy_q[~mask_q] = np.nan
ax.quiver(X[::skip, ::skip], Y[::skip, ::skip], Vx_q, Vy_q,
          color='navy', scale=15, alpha=0.8, width=0.004)

ax.add_patch(Circle((0, 0), 1.0, fill=False, ec='black', lw=2, ls='--'))
ax.add_patch(Circle((0, 0), 0.3, fill=True, fc='dimgray', ec='black', lw=1.5))
ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5); ax.set_aspect('equal')
ax.set_title('Evershed Flow (radial outflow)\nEvershed 흐름 (방사 유출)', fontsize=12)
ax.set_xlabel('x (spot radii)'); ax.set_ylabel('y (spot radii)')

# --- Panel 2: Magnetic field (from Hale paper) ---
ax = axes[1]
B0 = 2900
B_r = B0 * np.exp(-R**2 / (2 * 0.5**2))
B_r[R > 1.2] = 0

im2 = ax.pcolormesh(X, Y, B_r, cmap='inferno', vmin=0, vmax=3000, shading='auto')
plt.colorbar(im2, ax=ax, label='|B| (Gauss)', shrink=0.8)
ax.add_patch(Circle((0, 0), 1.0, fill=False, ec='white', lw=2, ls='--'))
ax.add_patch(Circle((0, 0), 0.3, fill=False, ec='white', lw=1.5))
ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5); ax.set_aspect('equal')
ax.set_title("Hale's Magnetic Field (1908)\nHale의 자기장 (1908)", fontsize=12)
ax.set_xlabel('x (spot radii)')

# --- Panel 3: Combined — flow vectors on B field ---
ax = axes[2]
im3 = ax.pcolormesh(X, Y, B_r, cmap='inferno', vmin=0, vmax=3000, shading='auto')
plt.colorbar(im3, ax=ax, label='|B| (Gauss)', shrink=0.8)
ax.quiver(X[::skip, ::skip], Y[::skip, ::skip], Vx_q, Vy_q,
          color='cyan', scale=15, alpha=0.9, width=0.004)
ax.add_patch(Circle((0, 0), 1.0, fill=False, ec='white', lw=2, ls='--'))
ax.add_patch(Circle((0, 0), 0.3, fill=False, ec='white', lw=1.5))
ax.set_xlim(-1.5, 1.5); ax.set_ylim(-1.5, 1.5); ax.set_aspect('equal')
ax.set_title('B field + Evershed flow = MHD\n자기장 + Evershed 흐름 = MHD', fontsize=12)
ax.set_xlabel('x (spot radii)')

plt.suptitle("From Hale (1908) + Evershed (1909): Sunspot as an MHD Structure\n"
             "Hale (1908) + Evershed (1909): 흑점 = MHD 구조체",
             fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Summary / 요약

| Part | 내용 / Content | Evershed (1909) 연결 / Connection |
|---|---|---|
| 1 | 방사 vs. 원형 운동의 Doppler 패턴 비교 | 논문의 핵심 논증 — 7가지 관측 사실의 물리적 근거 |
| 2 | Line-of-Nodes 테이블 재현 | 결정적 증거 — 평균 90°가 방사 운동 확정 |
| 3 | 합성 2D 스펙트럼의 선 기울기 | Evershed가 실제 관측한 "bent or inclined" 선 재현 |
| 4 | 자기장 + 흐름 결합 시각화 | Hale(1908) + Evershed(1909) = 흑점의 MHD 구조 |

**다음 논문 / Next paper**: Hale & Nicholson (1925) — 흑점 극성 법칙과 22년 자기 주기.
소용돌이 가설이 폐기된 후, Hale은 대규모 자기 주기 패턴으로 연구 방향을 전환합니다.

**Next paper**: Hale & Nicholson (1925) — Sunspot polarity law and the 22-year magnetic cycle.
After the vortex hypothesis was abandoned, Hale pivoted to large-scale magnetic cycle patterns.